# NB00 — Training Data Augmentation

**Purpose**: Synthetically expand the labelled dataset to give NB02/03 more training
samples and ensure every stratum has enough members for a stratified split.

**When to disable**: Set `USE_AUGMENTATION = False` once real survey data is available.
With that flag off this notebook is a passthrough — it adds the required columns and
writes `bt_scores_augmented.csv` unchanged, so NB01 onwards need no edits.

**How augmentation works**:
- Each labelled intersection gets `AUG_FACTOR` extra copies (default 2 → 3× data).
- Each copy stores an `aug_seed` that NB02/03 use to apply a reproducible random
  image transform (crop, flip, colour jitter) when extracting features.
- The BT score gets a small Gaussian perturbation (σ = `LABEL_NOISE_SD`) as label
  smoothing, clipped to [0, 1].
- Augmented copies of **validation** intersections are excluded from training to
  prevent leakage — the val set always contains original images only.

**Output**: `data/processed/bt_scores_augmented.csv`

---

**Imports** — Standard libraries only; no ML needed here.

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

**Configuration** — Set `USE_AUGMENTATION = False` when real survey data arrives; everything else can stay as-is.

In [23]:
# ---------------------------------------------------------------------------
# SWITCH — flip to False when real survey data is available
# ---------------------------------------------------------------------------
USE_AUGMENTATION = True

# ---------------------------------------------------------------------------
# CONFIGURATION — only relevant when USE_AUGMENTATION = True
# ---------------------------------------------------------------------------

# Extra copies per intersection (2 → 3x total training data)
AUG_FACTOR = 4

# Std of Gaussian noise added to bt_score for label smoothing.
# Small enough not to distort ranking; large enough to add variety.
LABEL_NOISE_SD = 0.02

# Fixed seed for reproducible aug_seeds and label noise
RANDOM_SEED = 42

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
CV_ROOT      = Path("..")
PROJECT_ROOT = CV_ROOT / ".."

BT_SCORES_CSV = CV_ROOT / "data" / "raw" / "bt_scores.csv"
LEGS_CSV      = PROJECT_ROOT / "intersections" / "data" / "processed" / "sampled_legs_directional_clean.csv"
OUTPUT_CSV    = CV_ROOT / "data" / "processed" / "bt_scores_augmented.csv"

(CV_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

print(f"USE_AUGMENTATION : {USE_AUGMENTATION}")
print(f"Input  : {BT_SCORES_CSV}  (exists={BT_SCORES_CSV.exists()})")
print(f"Legs   : {LEGS_CSV}  (exists={LEGS_CSV.exists()})")
print(f"Output : {OUTPUT_CSV}")

USE_AUGMENTATION : True
Input  : ..\data\raw\bt_scores.csv  (exists=True)
Legs   : ..\..\intersections\data\processed\sampled_legs_directional_clean.csv  (exists=True)
Output : ..\data\processed\bt_scores_augmented.csv


Load the original BT scores and add the bookkeeping columns that downstream notebooks expect.

In [24]:
# Load original BT labels
bt = pd.read_csv(BT_SCORES_CSV)

# Normalise column name — the BT pipeline exports "bt_score_normalized";
# rename to "bt_score" so all downstream code uses a single consistent name.
if "bt_score_normalized" in bt.columns and "bt_score" not in bt.columns:
    bt = bt.rename(columns={"bt_score_normalized": "bt_score"})

# Merge stratum columns from the legs CSV.
# The legs file has one row per sampled leg; dim_type/is_centrum etc. are
# intersection-level attributes so they are identical across legs — keep the first.
STRATUM_COLS = ["intersection_id", "leg_bearing", "dim_type", "dim_risk",
                "dim_priority", "dim_speed", "is_centrum"]
legs = (
    pd.read_csv(LEGS_CSV, usecols=STRATUM_COLS)
    .drop_duplicates(subset="intersection_id", keep="first")
)
bt = bt.merge(legs, on="intersection_id", how="left")

missing_stratum = bt["dim_type"].isna().sum()
if missing_stratum:
    print(f"WARNING: {missing_stratum} intersections have no legs entry — dim_type will be NaN.")

# Add columns that identify each row as an original (non-augmented) sample
bt["is_augmented"] = False
bt["aug_seed"]     = 0
bt["sample_id"]    = bt["intersection_id"].astype(str) + "_orig"

print(f"Loaded {len(bt)} original labelled intersections.")
print(f"Columns: {list(bt.columns)}")

Loaded 85 original labelled intersections.
Columns: ['intersection_id', 'bt_param_raw', 'bt_score', 'n_comparisons', 'n_wins', 'image_path', 'leg_bearing', 'dim_type', 'dim_risk', 'dim_priority', 'dim_speed', 'is_centrum', 'is_augmented', 'aug_seed', 'sample_id']


Create augmented copies if the switch is on, otherwise skip and keep only the originals.

In [25]:
if not USE_AUGMENTATION:
    # Passthrough — no augmentation, just write the originals with the new columns
    combined = bt.copy()
    print("USE_AUGMENTATION = False → passthrough, no copies created.")

else:
    rng = np.random.default_rng(RANDOM_SEED)
    aug_rows = []

    for _, row in bt.iterrows():
        for aug_i in range(1, AUG_FACTOR + 1):
            new_row = row.copy()

            # Unique seed used in NB02/03 to apply a deterministic random transform
            new_row["aug_seed"] = int(rng.integers(1, 100_000))

            # Small label noise keeps the score ordering but adds variety
            noise = rng.normal(0, LABEL_NOISE_SD)
            new_row["bt_score"] = float(np.clip(row["bt_score"] + noise, 0.0, 1.0))

            new_row["is_augmented"] = True
            new_row["sample_id"]    = f"{row['intersection_id']}_aug{aug_i}"
            aug_rows.append(new_row)

    aug_df   = pd.DataFrame(aug_rows)
    combined = pd.concat([bt, aug_df], ignore_index=True)

    n_orig = (combined["is_augmented"] == False).sum()
    n_aug  = (combined["is_augmented"] == True).sum()
    print(f"Original samples : {n_orig}")
    print(f"Augmented copies : {n_aug}  (×{AUG_FACTOR} per intersection)")
    print(f"Total rows       : {len(combined)}")

Original samples : 85
Augmented copies : 340  (×4 per intersection)
Total rows       : 425


Verify that the stratum distribution now has enough members per group for a stratified split.

In [26]:
# Check stratum counts on ORIGINALS only — the split will be based on these
originals = combined[combined["is_augmented"] == False].copy()
originals["dim_type"]   = originals["dim_type"].fillna("unknown")
originals["is_centrum"] = originals["is_centrum"].fillna(False).astype(str)
originals["strat_key"]  = originals["dim_type"].str.strip() + "_" + originals["is_centrum"]

key_counts = originals["strat_key"].value_counts()
print("Stratum counts (originals only):")
print(key_counts.to_string())
print()

# Warn about strata that still need collapsing (will be handled in NB01)
rare = key_counts[key_counts < 2]
if rare.empty:
    print("All strata have ≥ 2 members — stratified split will work.")
else:
    print(f"Strata with < 2 members (will be collapsed to 'other' in NB01): {list(rare.index)}")

Stratum counts (originals only):
strat_key
4+_False         43
T_False          23
4+_True          13
unknown_False     5
T_True            1

Strata with < 2 members (will be collapsed to 'other' in NB01): ['T_True']


Save the augmented (or passthrough) dataset to `data/processed/`.

In [27]:
# Save — NB01 onwards always reads from this file regardless of USE_AUGMENTATION
combined.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(combined)} rows to {OUTPUT_CSV}")
print(f"Columns: {list(combined.columns)}")

Saved 425 rows to ..\data\processed\bt_scores_augmented.csv
Columns: ['intersection_id', 'bt_param_raw', 'bt_score', 'n_comparisons', 'n_wins', 'image_path', 'leg_bearing', 'dim_type', 'dim_risk', 'dim_priority', 'dim_speed', 'is_centrum', 'is_augmented', 'aug_seed', 'sample_id']


## Summary

| | |
|---|---|
| `USE_AUGMENTATION` | `True` — disable when real survey data arrives |
| Original intersections | 68 |
| Total samples after augmentation | 204 (3×) |
| Label noise σ | 0.02 |
| Output | `data/processed/bt_scores_augmented.csv` |

**Next**: run NB01 to audit the augmented data and produce the train/val split.